In [60]:
import pandas as pd

### Loading data

In [61]:
df = pd.read_json('data.json')

In [62]:
df

,id,full_name,national_id,date_of_birth,email,city,country,occupation,net_monthly_income
0,1,James Anderson,666-45-1234,1982-03-12,j.anderson82@gmail.com,New York,USA,Software Engineer,8500
1,2,Robert Miller,666-45-9999,1985-11-20,r.miller@gmail.com,Chicago,USA,Systems Admin,7200
2,3,Linda Taylor,666-45-8888,1980-05-14,l.taylor@gmail.com,Miami,USA,DevOps,9100
3,4,Michael Brown,666-45-7777,1989-12-01,m.brown@gmail.com,Seattle,USA,QA,6800
4,5,Siddharth Gupta,1234-5678-12,1992-09-18,sid.gupta@infosys.in,Bangalore,India,Data Scientist,4500
5,6,Anjali Sharma,9999-8888-77,1995-03-30,anjali.s@infosys.in,Mumbai,India,ML Engineer,4200
6,7,Arjun Patel,1111-2222-33,1990-01-15,arjun.p@infosys.in,Delhi,India,Coder,4800
7,8,Priya Das,4444-5555-66,1999-11-20,priya.d@infosys.in,Pune,India,Analyst,4000
8,9,Rohan Singh,7777-8888-99,1993-04-10,rohan.s@infosys.in,Hyderabad,India,Support,3500
9,10,Sita Rao,0000-1111-22,1997-08-22,sita.r@infosys.in,Chennai,India,Admin,3200


### Base anonymization

##### Suppression - removing the data to be anonymize

In [63]:
def base_anonymization(input_df):
    df_base = input_df.copy()
    
    df_base = df_base.drop(columns=['full_name', 'national_id', 'id'])

    # Masking
    df_base['email'] = df_base['email'].apply(lambda x: "****@" + x.split('@')[1])
    
    return df_base

### Base results

In [64]:
df_base_output = base_anonymization(df)
df_base_output

,date_of_birth,email,city,country,occupation,net_monthly_income
0,1982-03-12,****@gmail.com,New York,USA,Software Engineer,8500
1,1985-11-20,****@gmail.com,Chicago,USA,Systems Admin,7200
2,1980-05-14,****@gmail.com,Miami,USA,DevOps,9100
3,1989-12-01,****@gmail.com,Seattle,USA,QA,6800
4,1992-09-18,****@infosys.in,Bangalore,India,Data Scientist,4500
5,1995-03-30,****@infosys.in,Mumbai,India,ML Engineer,4200
6,1990-01-15,****@infosys.in,Delhi,India,Coder,4800
7,1999-11-20,****@infosys.in,Pune,India,Analyst,4000
8,1993-04-10,****@infosys.in,Hyderabad,India,Support,3500
9,1997-08-22,****@infosys.in,Chennai,India,Admin,3200


### Good anonymization

In [ ]:
def anonymize_good(df):
    df_good = base_anonymization(df)
    
    # Generalization
    df_good['birth_decade'] = pd.to_datetime(df['date_of_birth']).dt.year.apply(lambda x: f"{(x // 10) * 10}s")
    
    # Bucketing
    bins = [0, 3000, 6000, 10000, float('inf')]
    labels = ['Low', 'Medium', 'High', 'Very high']
    df_good['income_bracket'] = pd.cut(df['net_monthly_income'], bins=bins, labels=labels)
    
    return df_good.drop(columns=['date_of_birth', 'city', 'net_monthly_income'])

### Good results

In [66]:
df_good = anonymize_good(df)
df_good

,email,country,occupation,birth_decade,income_bracket
0,****@gmail.com,USA,Software Engineer,1980s,High
1,****@gmail.com,USA,Systems Admin,1980s,High
2,****@gmail.com,USA,DevOps,1980s,High
3,****@gmail.com,USA,QA,1980s,High
4,****@infosys.in,India,Data Scientist,1990s,Medium
5,****@infosys.in,India,ML Engineer,1990s,Medium
6,****@infosys.in,India,Coder,1990s,Medium
7,****@infosys.in,India,Analyst,1990s,Medium
8,****@infosys.in,India,Support,1990s,Medium
9,****@infosys.in,India,Admin,1990s,Medium


### K-anonymity check

The higher k value the better privacy but at the same time data utility is worse. 

In [67]:
k_groups = df_good.groupby(['birth_decade', 'country']).size().reset_index(name='k_count')
min_k = k_groups['k_count'].min()

print(k_groups)

  birth_decade  country  k_count
0        1960s   Sweden        2
1        1970s    Japan        2
2        1980s    Italy        3
3        1980s      USA        4
4        1990s  Germany        3
5        1990s    India        6


### Data utility metrics

### Mean distortion

In [68]:
original_mean = df['net_monthly_income'].mean()

group_means = {'Low': 1500, 'Medium': 4500, 'High': 8000, 'Very high': 120000}
anonymized_approx_mean = df_good['income_bracket'].map(group_means).astype(float).mean()

distortion = abs(original_mean - anonymized_approx_mean) / original_mean * 100
print(f"Original mean: {original_mean}")
print(f"Anonymized mean: {anonymized_approx_mean}")
print(f"Lost of income mean: {distortion:.2f}%")

Original mean: 5910.0
Anonymized mean: 11325.0
Lost of income mean: 91.62%


### Query accuracy
How many people earns more than 5000?

In [69]:
count_orig = len(df[df['net_monthly_income'] > 5000])

count_anon = len(df_good[df_good['income_bracket'].isin(['High', 'Very high'])])

accuracy = (count_anon / count_orig) * 100
print(f'Original count: {count_orig}')
print(f'Anonymized count: {count_anon}')
print(f"Accuracy: {accuracy}%")

Original count: 11
Anonymized count: 7
Accuracy: 63.63636363636363%
